In [1]:
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q kaggle

import os
import random
import shutil
import subprocess
from pathlib import Path
from collections import Counter

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image

if IN_COLAB:
    from google.colab import files  # type: ignore
else:
    files = None

NOTEBOOK_DIR = Path.cwd()
print("Running from:", NOTEBOOK_DIR)

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 362, in run
    resolver = self.make_resolver(
               ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 177, in make_resolver
    return pip._internal.resolution.resolvelib.resolver.Resolver(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 58, in __init__
    self.factory = Factory(
                   ^^^^^^^^
  File "/usr/local/lib/py

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


In [3]:
if IN_COLAB:
    from google.colab import userdata  # type: ignore

    KAGGLE_USERNAME = userdata.get("KAGGLE_USERNAME")
    KAGGLE_KEY = userdata.get("KAGGLE_KEY")
else:
    KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME")
    KAGGLE_KEY = os.environ.get("KAGGLE_KEY")

if KAGGLE_USERNAME and KAGGLE_KEY:
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
    os.environ["KAGGLE_KEY"] = KAGGLE_KEY
    print("Kaggle API credentials loaded.")
else:
    print(
        "Kaggle credentials were not found. "
        "Set KAGGLE_USERNAME and KAGGLE_KEY, or place kaggle.json in the standard Kaggle config path."
    )

Kaggle API credentials loaded.


In [4]:
KAGGLE_DATASET = "ninadaithal/imagesoasis"

RAW_DATA_DIR = NOTEBOOK_DIR / "oasis_raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

if any(RAW_DATA_DIR.rglob("*")):
    print("Using existing raw dataset at:", RAW_DATA_DIR)
else:
    print("Downloading dataset:", KAGGLE_DATASET)
    subprocess.run(
        [
            "kaggle", "datasets", "download",
            "-d", KAGGLE_DATASET,
            "-p", str(RAW_DATA_DIR),
            "--unzip"
        ],
        check=True
    )
    print("Dataset downloaded to:", RAW_DATA_DIR)

Dataset downloaded to: /content/oasis_raw


In [5]:
def show_folder_tree(root_dir, max_depth=3):
    root_dir = Path(root_dir)
    print(f"\nFolder tree for: {root_dir}\n")

    for root, dirs, file_list in os.walk(root_dir):
        root_path = Path(root)
        depth = len(root_path.relative_to(root_dir).parts)

        if depth > max_depth:
            continue

        indent = "    " * depth
        print(f"{indent}{root_path.name}/")

        if depth == max_depth:
            continue


show_folder_tree(RAW_DATA_DIR, max_depth=3)



Folder tree for: /content/oasis_raw

oasis_raw/
    Data/
        Mild Dementia/
        Moderate Dementia/
        Very mild Dementia/
        Non Demented/


In [6]:
READY_DATA_DIR = NOTEBOOK_DIR / "oasis_ready"
VAL_RATIO = 0.2
SEED = 42

VALID_EXTENSIONS = [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".npy"]
CLASS_ORDER = ["Non Demented", "Very mild Dementia", "Mild Dementia"]
CLASS_TO_INDEX = {class_name: idx for idx, class_name in enumerate(CLASS_ORDER)}


def clean_class_name(name):
    """
    Converts Kaggle class folder names into the same three-class label space as
    the main U-Net CVAE experiment. Moderate dementia is intentionally excluded.
    """
    normalized = name.lower().strip()
    normalized = normalized.replace(" ", "_")
    normalized = normalized.replace("-", "_")
    normalized = normalized.replace("__", "_")

    if "moderate" in normalized:
        return None
    if "non" in normalized and "dement" in normalized:
        return "Non Demented"
    if "very" in normalized and "mild" in normalized:
        return "Very mild Dementia"
    if "mild" in normalized:
        return "Mild Dementia"

    return None


def find_class_folders(source_dir):
    """
    Finds folders that directly contain image files.
    """
    source_dir = Path(source_dir)
    class_folders = []

    for folder in source_dir.rglob("*"):
        if folder.is_dir():
            image_files = [
                f for f in folder.iterdir()
                if f.is_file() and f.suffix.lower() in VALID_EXTENSIONS
            ]
            if len(image_files) > 0:
                class_folders.append(folder)

    return class_folders


class_folders = find_class_folders(RAW_DATA_DIR)
usable_class_folders = []

print("Detected class folders:")
for folder in class_folders:
    class_name = clean_class_name(folder.name)
    print("-", folder, "->", class_name if class_name is not None else "ignored")
    if class_name is not None:
        usable_class_folders.append((folder, class_name))

if len(usable_class_folders) == 0:
    raise ValueError("No usable image folders were detected. Expected Non Demented, Very mild Dementia, and Mild Dementia classes.")

# Remove old prepared dataset
if READY_DATA_DIR.exists():
    shutil.rmtree(READY_DATA_DIR)

random.seed(SEED)

for class_folder, class_name in usable_class_folders:
    image_files = [
        f for f in class_folder.iterdir()
        if f.is_file() and f.suffix.lower() in VALID_EXTENSIONS
    ]

    random.shuffle(image_files)

    val_count = int(len(image_files) * VAL_RATIO)
    val_files = image_files[:val_count]
    train_files = image_files[val_count:]

    train_out_dir = READY_DATA_DIR / "train" / class_name
    val_out_dir = READY_DATA_DIR / "val" / class_name

    train_out_dir.mkdir(parents=True, exist_ok=True)
    val_out_dir.mkdir(parents=True, exist_ok=True)

    for file_path in train_files:
        shutil.copy(file_path, train_out_dir / file_path.name)

    for file_path in val_files:
        shutil.copy(file_path, val_out_dir / file_path.name)

    print(f"{class_name}: train={len(train_files)}, val={len(val_files)}")

missing_classes = [class_name for class_name in CLASS_ORDER if not (READY_DATA_DIR / "train" / class_name).exists()]
if missing_classes:
    raise ValueError(f"Missing required classes after preparation: {missing_classes}")

print("Prepared dataset saved to:", READY_DATA_DIR)
show_folder_tree(READY_DATA_DIR, max_depth=2)

Detected class folders:
- /content/oasis_raw/Data/Mild Dementia -> Mild Dementia
- /content/oasis_raw/Data/Moderate Dementia -> ignored
- /content/oasis_raw/Data/Very mild Dementia -> Very mild Dementia
- /content/oasis_raw/Data/Non Demented -> Non Demented
Mild Dementia: train=4002, val=1000
Very mild Dementia: train=10980, val=2745
Non Demented: train=53778, val=13444
Prepared dataset saved to: /content/oasis_ready

Folder tree for: /content/oasis_ready

oasis_ready/
    train/
        Mild Dementia/
        Very mild Dementia/
        Non Demented/
    val/
        Mild Dementia/
        Very mild Dementia/
        Non Demented/


In [7]:
def count_images_per_class(split_dir):
    split_dir = Path(split_dir)
    counts = {}

    for class_folder in sorted(split_dir.iterdir()):
        if class_folder.is_dir():
            files = [
                f for f in class_folder.iterdir()
                if f.is_file() and f.suffix.lower() in VALID_EXTENSIONS
            ]
            counts[class_folder.name] = len(files)

    return counts


print("\nTrain distribution:")
print(count_images_per_class(READY_DATA_DIR / "train"))

print("\nValidation distribution:")
print(count_images_per_class(READY_DATA_DIR / "val"))



Train distribution:
{'Mild Dementia': 4002, 'Non Demented': 53778, 'Very mild Dementia': 10980}

Validation distribution:
{'Mild Dementia': 1000, 'Non Demented': 13444, 'Very mild Dementia': 2745}


In [8]:
class CFG:
    DATA_DIR = str(READY_DATA_DIR)
    TRAIN_DIR = str(READY_DATA_DIR / "train")
    VAL_DIR = str(READY_DATA_DIR / "val")

    # Match the main U-Net CVAE experiment for fair baseline comparison.
    IMG_SIZE = 224
    CHANNELS = 1
    NUM_CLASSES = 3

    BATCH_SIZE = 32
    NUM_EPOCHS = 10
    LEARNING_RATE = 1e-4

    LATENT_DIM = 128

    # Match the main U-Net CVAE KL regularization strength.
    BETA = 1.0

    NUM_WORKERS = 2
    SEED = 42

    OUTPUT_DIR = str(NOTEBOOK_DIR / "outputs_vanilla_cvae")
    CHECKPOINT_DIR = str(NOTEBOOK_DIR / "checkpoints_vanilla_cvae")


os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
os.makedirs(CFG.CHECKPOINT_DIR, exist_ok=True)

In [9]:
class MRIDataset(Dataset):

    valid_extensions = VALID_EXTENSIONS

    def __init__(self, root_dir, class_to_idx=None, img_size=224):
        self.root_dir = Path(root_dir)
        self.img_size = img_size

        if not self.root_dir.exists():
            raise FileNotFoundError(f"Dataset folder not found: {self.root_dir}")

        if class_to_idx is None:
            self.class_to_idx = CLASS_TO_INDEX.copy()
        else:
            self.class_to_idx = class_to_idx

        self.idx_to_class = {idx: class_name for class_name, idx in self.class_to_idx.items()}
        self.samples = []

        for class_name in CLASS_ORDER:
            if class_name not in self.class_to_idx:
                continue

            class_folder = self.root_dir / class_name
            if not class_folder.exists():
                raise FileNotFoundError(f"Required class folder not found: {class_folder}")

            label = self.class_to_idx[class_name]
            for file_path in class_folder.rglob("*"):
                if file_path.suffix.lower() in self.valid_extensions:
                    self.samples.append((str(file_path), label))

        if len(self.samples) == 0:
            raise ValueError(f"No images found in {self.root_dir}")

        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.samples)

    def load_image(self, file_path):
        file_path = Path(file_path)

        if file_path.suffix.lower() == ".npy":
            arr = np.load(file_path)
            arr = np.squeeze(arr)

            # If a 3D volume is found, take the middle slice.
            if arr.ndim == 3:
                middle = arr.shape[0] // 2
                arr = arr[middle]

            arr = arr.astype(np.float32)
            arr_min = arr.min()
            arr_max = arr.max()

            if arr_max > arr_min:
                arr = (arr - arr_min) / (arr_max - arr_min)
            else:
                arr = np.zeros_like(arr)

            arr = (arr * 255).astype(np.uint8)
            image = Image.fromarray(arr).convert("L")

        else:
            image = Image.open(file_path).convert("L")

        return image

    def __getitem__(self, idx):
        file_path, label = self.samples[idx]
        image = self.load_image(file_path)
        image = self.transform(image)
        label = torch.tensor(label, dtype=torch.long)
        return image, label


train_dataset = MRIDataset(CFG.TRAIN_DIR, img_size=CFG.IMG_SIZE)
val_dataset = MRIDataset(
    CFG.VAL_DIR,
    class_to_idx=train_dataset.class_to_idx,
    img_size=CFG.IMG_SIZE
)

NUM_CLASSES = len(train_dataset.class_to_idx)
assert NUM_CLASSES == CFG.NUM_CLASSES, f"Expected {CFG.NUM_CLASSES} classes, found {NUM_CLASSES}"

print("Class mapping:")
print(train_dataset.class_to_idx)
print("Number of classes:", NUM_CLASSES)
print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=True,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True
)

Class mapping:
{'Non Demented': 0, 'Very mild Dementia': 1, 'Mild Dementia': 2}
Number of classes: 3
Train samples: 68760
Validation samples: 17189


In [10]:
images, labels = next(iter(train_loader))
print("\nOne batch image shape:", images.shape)
print("One batch label shape:", labels.shape)
print("Example labels:", labels[:10])

sample_grid_path = os.path.join(CFG.OUTPUT_DIR, "sample_training_images.png")
save_image(images[:16], sample_grid_path, nrow=8)
print("Saved sample image grid:", sample_grid_path)


One batch image shape: torch.Size([32, 1, 224, 224])
One batch label shape: torch.Size([32])
Example labels: tensor([0, 0, 2, 0, 0, 1, 0, 1, 0, 0])
Saved sample image grid: /content/outputs_vanilla_cvae/sample_training_images.png


In [11]:
def label_to_onehot(labels, num_classes):
    return F.one_hot(labels, num_classes=num_classes).float()


def make_label_map(labels, num_classes, height, width):
    """
    Converts labels into spatial label maps.

    Image shape:     [B, 1, H, W]
    Label map shape: [B, C, H, W]
    Combined input:  [B, 1 + C, H, W]
    """
    onehot = label_to_onehot(labels, num_classes)
    label_map = onehot[:, :, None, None]
    label_map = label_map.repeat(1, 1, height, width)
    return label_map

In [12]:
class VanillaCVAE(nn.Module):

    def __init__(self, img_size=224, channels=1, num_classes=3, latent_dim=128):
        super().__init__()
        self.img_size = img_size
        self.channels = channels
        self.num_classes = num_classes
        self.latent_dim = latent_dim

        encoder_input_channels = channels + num_classes

        self.encoder = nn.Sequential(
            nn.Conv2d(encoder_input_channels, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
        )

        self.feature_size = img_size // 16
        self.flatten_dim = 256 * self.feature_size * self.feature_size

        self.fc_mu = nn.Linear(self.flatten_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.flatten_dim, latent_dim)

        decoder_input_dim = latent_dim + num_classes
        self.decoder_input = nn.Linear(decoder_input_dim, self.flatten_dim)

        # Correct channel flow:
        # 256 -> 128 -> 64 -> 32 -> 1
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(32, channels, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid()
        )


    def encode(self, x, labels):
        batch_size, _, height, width = x.shape
        label_map = make_label_map(labels, self.num_classes, height, width).to(x.device)

        encoder_input = torch.cat([x, label_map], dim=1)
        features = self.encoder(encoder_input)
        features = features.view(batch_size, -1)

        mu = self.fc_mu(features)
        logvar = self.fc_logvar(features)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z

    def decode(self, z, labels):
        onehot = label_to_onehot(labels, self.num_classes).to(z.device)
        decoder_input = torch.cat([z, onehot], dim=1)

        x = self.decoder_input(decoder_input)
        x = x.view(-1, 256, self.feature_size, self.feature_size)

        recon = self.decoder(x)
        return recon

    def forward(self, x, labels):
        mu, logvar = self.encode(x, labels)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z, labels)
        return recon, mu, logvar

model = VanillaCVAE(
    img_size=CFG.IMG_SIZE,
    channels=CFG.CHANNELS,
    num_classes=NUM_CLASSES,
    latent_dim=CFG.LATENT_DIM
).to(DEVICE)

print("\nModel created successfully.")
print(model)


Model created successfully.
VanillaCVAE(
  (encoder): Sequential(
    (0): Conv2d(4, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.2, inplace=True)
    (3): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): LeakyReLU(negative_slope=0.2, inplace=True)
    (6): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (7): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): LeakyReLU(negative_slope=0.2, inplace=True)
    (9): Conv2d(128, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (10): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): LeakyReLU(negative_slope=0.2, inplace=True)
  )
  (fc_mu): Linear(in_features=50176, out_features=128, 

In [13]:
def cvae_loss(recon, x, mu, logvar, beta=1.0):
    # Match the main U-Net CVAE reconstruction objective, without classifier guidance.
    recon_loss = F.binary_cross_entropy(recon, x, reduction="mean")
    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    total_loss = recon_loss + beta * kl_loss
    return total_loss, recon_loss, kl_loss


optimizer = torch.optim.Adam(model.parameters(), lr=CFG.LEARNING_RATE)

In [14]:
def train_one_epoch(model, dataloader, optimizer):
    model.train()

    total_loss_sum = 0.0
    recon_loss_sum = 0.0
    kl_loss_sum = 0.0

    for images, labels in dataloader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        recon, mu, logvar = model(images, labels)
        loss, recon_loss, kl_loss = cvae_loss(
            recon, images, mu, logvar, beta=CFG.BETA
        )

        loss.backward()
        optimizer.step()

        total_loss_sum += loss.item()
        recon_loss_sum += recon_loss.item()
        kl_loss_sum += kl_loss.item()

    n = len(dataloader)
    return total_loss_sum / n, recon_loss_sum / n, kl_loss_sum / n

@torch.no_grad()
def validate_one_epoch(model, dataloader):
    model.eval()

    total_loss_sum = 0.0
    recon_loss_sum = 0.0
    kl_loss_sum = 0.0

    for images, labels in dataloader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        recon, mu, logvar = model(images, labels)
        loss, recon_loss, kl_loss = cvae_loss(
            recon, images, mu, logvar, beta=CFG.BETA
        )

        total_loss_sum += loss.item()
        recon_loss_sum += recon_loss.item()
        kl_loss_sum += kl_loss.item()

    n = len(dataloader)
    return total_loss_sum / n, recon_loss_sum / n, kl_loss_sum / n


In [15]:
@torch.no_grad()
def save_reconstruction_examples(model, dataloader, epoch, max_images=8):
    """
    Saves original and reconstructed images.

    Top row: original images
    Bottom row: reconstructed images
    """
    model.eval()

    images, labels = next(iter(dataloader))
    images = images[:max_images].to(DEVICE)
    labels = labels[:max_images].to(DEVICE)

    recon, _, _ = model(images, labels)

    comparison = torch.cat([images.cpu(), recon.cpu()], dim=0)
    save_path = os.path.join(CFG.OUTPUT_DIR, f"reconstruction_epoch_{epoch:03d}.png")
    save_image(comparison, save_path, nrow=max_images)

    print("Saved reconstruction example:", save_path)

In [16]:
@torch.no_grad()
def generate_samples_by_class(model, num_samples_per_class=4, file_name="generated_by_class.png"):
    """
    Generates new MRI-like images from random latent vectors.

    Each row represents one dementia class.
    """
    model.eval()

    all_generated = []

    for class_idx in range(NUM_CLASSES):
        z = torch.randn(num_samples_per_class, CFG.LATENT_DIM).to(DEVICE)
        labels = torch.full(
            size=(num_samples_per_class,),
            fill_value=class_idx,
            dtype=torch.long
        ).to(DEVICE)

        generated = model.decode(z, labels)
        all_generated.append(generated.cpu())

    all_generated = torch.cat(all_generated, dim=0)

    save_path = os.path.join(CFG.OUTPUT_DIR, file_name)
    save_image(all_generated, save_path, nrow=num_samples_per_class)
    print("Saved generated samples:", save_path)

In [17]:
best_val_loss = float("inf")
train_history = []
val_history = []

for epoch in range(1, CFG.NUM_EPOCHS + 1):
    train_loss, train_recon, train_kl = train_one_epoch(model, train_loader, optimizer)
    val_loss, val_recon, val_kl = validate_one_epoch(model, val_loader)

    train_history.append({
        "epoch": epoch,
        "loss": train_loss,
        "recon": train_recon,
        "kl": train_kl,
    })

    val_history.append({
        "epoch": epoch,
        "loss": val_loss,
        "recon": val_recon,
        "kl": val_kl,
    })

    print(
        f"Epoch [{epoch:03d}/{CFG.NUM_EPOCHS}] "
        f"Train Loss: {train_loss:.6f} | Train Recon: {train_recon:.6f} | Train KL: {train_kl:.6f} || "
        f"Val Loss: {val_loss:.6f} | Val Recon: {val_recon:.6f} | Val KL: {val_kl:.6f}"
    )

    if epoch % 5 == 0:
        save_reconstruction_examples(model, val_loader, epoch)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        checkpoint_path = os.path.join(CFG.CHECKPOINT_DIR, "best_vanilla_cvae.pth")

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": val_loss,
            "class_to_idx": train_dataset.class_to_idx,
            "idx_to_class": train_dataset.idx_to_class,
            "config": {
                "img_size": CFG.IMG_SIZE,
                "channels": CFG.CHANNELS,
                "num_classes": NUM_CLASSES,
                "latent_dim": CFG.LATENT_DIM,
                "beta": CFG.BETA,
                "reconstruction_loss": "bce",
            }
        }, checkpoint_path)

        print("Saved best checkpoint:", checkpoint_path)

# Save final reconstruction and generated samples
save_reconstruction_examples(model, val_loader, CFG.NUM_EPOCHS, max_images=8)
generate_samples_by_class(model, num_samples_per_class=4)

print("Training complete.")
print("Best validation loss:", best_val_loss)
print("Outputs saved to:", CFG.OUTPUT_DIR)
print("Checkpoints saved to:", CFG.CHECKPOINT_DIR)

Epoch [001/10] Train Loss: 3531721.969016 | Train Recon: 0.398627 | Train KL: 3531721.570810 || Val Loss: 7.067890 | Val Recon: 0.446546 | Val KL: 6.621344
Saved best checkpoint: /content/checkpoints_vanilla_cvae/best_vanilla_cvae.pth
Epoch [002/10] Train Loss: 6.008209 | Train Recon: 0.368757 | Train KL: 5.639452 || Val Loss: 2.245946 | Val Recon: 0.364723 | Val KL: 1.881223
Saved best checkpoint: /content/checkpoints_vanilla_cvae/best_vanilla_cvae.pth
Epoch [003/10] Train Loss: 2.499083 | Train Recon: 0.363710 | Train KL: 2.135372 || Val Loss: 1.722633 | Val Recon: 0.360527 | Val KL: 1.362106
Saved best checkpoint: /content/checkpoints_vanilla_cvae/best_vanilla_cvae.pth
Epoch [004/10] Train Loss: 1.985415 | Train Recon: 0.362108 | Train KL: 1.623307 || Val Loss: 1.425680 | Val Recon: 0.361038 | Val KL: 1.064642
Saved best checkpoint: /content/checkpoints_vanilla_cvae/best_vanilla_cvae.pth
Epoch [005/10] Train Loss: 398.838131 | Train Recon: 0.361644 | Train KL: 398.476495 || Val Loss

In [18]:
# --- Evaluation Metrics: MSE, PSNR, SSIM (same as unet/src/mid_unet_cvae/metrics.py) ---

@torch.no_grad()
def compute_mse(recon, target):
    return F.mse_loss(recon, target, reduction="mean")


@torch.no_grad()
def compute_psnr(recon, target, max_value=1.0):
    value = compute_mse(recon, target).clamp_min(1e-12)
    return 20 * torch.log10(torch.tensor(max_value, device=recon.device)) - 10 * torch.log10(value)


@torch.no_grad()
def compute_ssim(recon, target):
    """Batch-level global SSIM for grayscale tensors in [0, 1]."""
    c1 = 0.01 ** 2
    c2 = 0.03 ** 2
    dims = (1, 2, 3)
    mu_x = recon.mean(dim=dims)
    mu_y = target.mean(dim=dims)
    sigma_x = ((recon - mu_x[:, None, None, None]) ** 2).mean(dim=dims)
    sigma_y = ((target - mu_y[:, None, None, None]) ** 2).mean(dim=dims)
    sigma_xy = (
        (recon - mu_x[:, None, None, None])
        * (target - mu_y[:, None, None, None])
    ).mean(dim=dims)
    numerator = (2 * mu_x * mu_y + c1) * (2 * sigma_xy + c2)
    denominator = (mu_x.pow(2) + mu_y.pow(2) + c1) * (sigma_x + sigma_y + c2)
    return (numerator / denominator.clamp_min(1e-12)).mean()


def eval_kl_divergence(mu, logvar):
    return -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

In [19]:
# --- Run Evaluation on Val Set (MSE, PSNR, SSIM, Val Loss) ---

# Load best checkpoint for evaluation
best_ckpt_path = os.path.join(CFG.CHECKPOINT_DIR, "best_vanilla_cvae.pth")
best_ckpt = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(best_ckpt["model_state_dict"])
model.eval()
print("Loaded best checkpoint from epoch:", best_ckpt.get("epoch", "?"))

mse_total = 0.0
psnr_total = 0.0
ssim_total = 0.0
bce_total = 0.0
kl_total_eval = 0.0
num_images = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        recon, mu, logvar = model(images, labels)
        bs = images.size(0)

        mse_total += compute_mse(recon, images).item() * bs
        psnr_total += compute_psnr(recon, images).item() * bs
        ssim_total += compute_ssim(recon, images).item() * bs
        bce_total += F.binary_cross_entropy(recon, images, reduction="mean").item() * bs
        kl_total_eval += eval_kl_divergence(mu, logvar).item() * bs
        num_images += bs

print(f"\n=== Vanilla CVAE Evaluation (Val Set) ===")
print(f"Images: {num_images}")
print(f"MSE: {mse_total / num_images:.6f}")
print(f"PSNR: {psnr_total / num_images:.4f}")
print(f"SSIM: {ssim_total / num_images:.4f}")
print(f"Recon Loss (BCE): {bce_total / num_images:.6f}")
print(f"KL Loss: {kl_total_eval / num_images:.6f}")
print(f"Val Loss (BCE + KL): {(bce_total + kl_total_eval) / num_images:.6f}")

Loaded best checkpoint from epoch: 8

=== Vanilla CVAE Evaluation (Val Set) ===
Images: 17189
MSE: 0.012198
PSNR: 19.1788
SSIM: 0.7504
Recon Loss (BCE): 0.362002
KL Loss: 0.889778
Val Loss (BCE + KL): 1.251780


In [20]:
# --- Classifier Definition (same as unet/src/mid_unet_cvae/models.py) ---
# This is the frozen classifier trained on ORIGINAL MRI images.
# It was NOT trained on any reconstructed images from either model.
# UNet used this classifier as part of its training loss (classifier guidance),
# while Vanilla never saw it during training.

def conv_down(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
    )


class SmallMRIClassifier(nn.Module):
    """Lightweight grayscale classifier used as the frozen guide."""

    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            conv_down(1, 32),
            conv_down(32, 64),
            conv_down(64, 128),
            conv_down(128, 256),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=0.2),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.head(self.features(x))


def normalize_for_classifier(images, mean=0.456, std=0.224):
    return (images - mean) / std


def format_confusion_matrix(matrix):
    rows = ["[" + ", ".join(str(int(v)) for v in row.tolist()) + "]" for row in matrix]
    return "\n".join(rows)


print("Classifier and helper functions defined.")

Classifier and helper functions defined.


In [21]:
# --- Train Vanilla Classifier on Original MRI Images ---
# This creates best_classifier.pth for evaluating Vanilla CVAE reconstructions.

import os
import torch
import torch.nn as nn
import torch.optim as optim

CLASSIFIER_CHECKPOINT_DIR = "/content/checkpoints_vanilla_classifier"
os.makedirs(CLASSIFIER_CHECKPOINT_DIR, exist_ok=True)

CLASSIFIER_CHECKPOINT = os.path.join(
    CLASSIFIER_CHECKPOINT_DIR,
    "best_classifier.pth"
)

classifier = SmallMRIClassifier(num_classes=CFG.NUM_CLASSES).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer_cls = optim.Adam(classifier.parameters(), lr=1e-4)

NUM_CLASSIFIER_EPOCHS = 10
best_val_acc = 0.0

for epoch in range(1, NUM_CLASSIFIER_EPOCHS + 1):
    classifier.train()

    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE).long()

        images = normalize_for_classifier(images)

        optimizer_cls.zero_grad()

        logits = classifier(images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer_cls.step()

        train_loss_sum += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)

        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total

    classifier.eval()

    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE).long()

            images = normalize_for_classifier(images)

            logits = classifier(images)
            loss = criterion(logits, labels)

            val_loss_sum += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)

            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total

    print(
        f"Epoch [{epoch:03d}/{NUM_CLASSIFIER_EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} || "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc

        torch.save(
            {
                "epoch": epoch,
                "num_classes": CFG.NUM_CLASSES,
                "model_state_dict": classifier.state_dict(),
                "optimizer_state_dict": optimizer_cls.state_dict(),
                "best_val_acc": best_val_acc,
                "class_to_idx": train_dataset.class_to_idx,
                "idx_to_class": train_dataset.idx_to_class,
            },
            CLASSIFIER_CHECKPOINT,
        )

        print("Saved best classifier:", CLASSIFIER_CHECKPOINT)

print("Finished classifier training.")
print("Best classifier validation accuracy:", best_val_acc)

Epoch [001/10] Train Loss: 0.4362 | Train Acc: 0.8219 || Val Loss: 0.7709 | Val Acc: 0.6334
Saved best classifier: /content/checkpoints_vanilla_classifier/best_classifier.pth
Epoch [002/10] Train Loss: 0.2466 | Train Acc: 0.9102 || Val Loss: 0.1558 | Val Acc: 0.9468
Saved best classifier: /content/checkpoints_vanilla_classifier/best_classifier.pth
Epoch [003/10] Train Loss: 0.1236 | Train Acc: 0.9632 || Val Loss: 0.1097 | Val Acc: 0.9617
Saved best classifier: /content/checkpoints_vanilla_classifier/best_classifier.pth
Epoch [004/10] Train Loss: 0.0659 | Train Acc: 0.9825 || Val Loss: 0.0593 | Val Acc: 0.9814
Saved best classifier: /content/checkpoints_vanilla_classifier/best_classifier.pth
Epoch [005/10] Train Loss: 0.0404 | Train Acc: 0.9898 || Val Loss: 0.1173 | Val Acc: 0.9532
Epoch [006/10] Train Loss: 0.0275 | Train Acc: 0.9934 || Val Loss: 0.1016 | Val Acc: 0.9570
Epoch [007/10] Train Loss: 0.0244 | Train Acc: 0.9936 || Val Loss: 0.0177 | Val Acc: 0.9952
Saved best classifier: /

In [22]:
# --- Classifier Evaluation: Accuracy, Confidence, Confusion Matrix ---
#
# IMPORTANT: You need the trained classifier checkpoint.
# Upload best_classifier.pth from unet/checkpoints/classifier/
# or set CLASSIFIER_CHECKPOINT to the correct path.

# if IN_COLAB:
#     import os
#     if not os.path.exists("best_classifier.pth"):
#         print("Please upload best_classifier.pth")
#         uploaded = files.upload()
#     CLASSIFIER_CHECKPOINT = "best_classifier.pth"
# else:
#     CLASSIFIER_CHECKPOINT = "best_classifier.pth"  # Change this path as needed

CLASSIFIER_CHECKPOINT = "/content/checkpoints_vanilla_classifier/best_classifier.pth"

# Load frozen classifier
cls_ckpt = torch.load(CLASSIFIER_CHECKPOINT, map_location=DEVICE, weights_only=False)
classifier = SmallMRIClassifier(
    num_classes=cls_ckpt.get("num_classes", 3)
).to(DEVICE)
classifier.load_state_dict(cls_ckpt["model_state_dict"])
classifier.eval()
for param in classifier.parameters():
    param.requires_grad = False
print("Loaded classifier checkpoint.")

# Load best vanilla checkpoint
best_ckpt_path = os.path.join(CFG.CHECKPOINT_DIR, "best_vanilla_cvae.pth")
best_ckpt = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(best_ckpt["model_state_dict"])
model.eval()
print("Loaded best Vanilla CVAE from epoch:", best_ckpt.get("epoch", "?"))

# Evaluate
correct = 0
total = 0
confidence_sum = 0.0
confusion = torch.zeros(CFG.NUM_CLASSES, CFG.NUM_CLASSES, dtype=torch.long)

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        recon, mu, logvar = model(images, labels)
        bs = images.size(0)

        logits = classifier(normalize_for_classifier(recon))
        probs = logits.softmax(dim=1)
        preds = logits.argmax(dim=1)

        confidence = probs.gather(1, labels.view(-1, 1)).mean().item()
        confidence_sum += confidence * bs
        correct += (preds == labels).sum().item()
        total += bs

        for label, pred in zip(labels.cpu().view(-1), preds.cpu().view(-1)):
            confusion[int(label), int(pred)] += 1

print(f"\n=== Vanilla CVAE Classifier Evaluation (Val Set) ===")
print(f"Images: {total}")
print(f"Target confidence: {confidence_sum / total:.4f}")
print(f"Classifier accuracy on reconstructions: {correct / total:.4f}")
print(f"Confusion matrix:")
print(format_confusion_matrix(confusion))

Loaded classifier checkpoint.
Loaded best Vanilla CVAE from epoch: 8

=== Vanilla CVAE Classifier Evaluation (Val Set) ===
Images: 17189
Target confidence: 0.7821
Classifier accuracy on reconstructions: 0.7821
Confusion matrix:
[13444, 0, 0]
[2745, 0, 0]
[1000, 0, 0]


In [27]:
outputs_zip = NOTEBOOK_DIR / "outputs_vanilla_cvae.zip"
checkpoints_zip = NOTEBOOK_DIR / "checkpoints_vanilla_cvae.zip"

if IN_COLAB:
    !zip -r {outputs_zip} {CFG.OUTPUT_DIR}
    !zip -r {checkpoints_zip} {CFG.CHECKPOINT_DIR}
else:
    shutil.make_archive(str(outputs_zip.with_suffix("")), "zip", CFG.OUTPUT_DIR)
    shutil.make_archive(str(checkpoints_zip.with_suffix("")), "zip", CFG.CHECKPOINT_DIR)

print("Archives written to:")
print(outputs_zip)
print(checkpoints_zip)

  adding: content/outputs_vanilla_cvae/ (stored 0%)
  adding: content/outputs_vanilla_cvae/generated_by_class.png (deflated 0%)
  adding: content/outputs_vanilla_cvae/sample_training_images.png (deflated 1%)
  adding: content/outputs_vanilla_cvae/reconstruction_epoch_010.png (deflated 0%)
  adding: content/outputs_vanilla_cvae/reconstruction_epoch_005.png (deflated 0%)
  adding: content/checkpoints_vanilla_cvae/ (stored 0%)
  adding: content/checkpoints_vanilla_cvae/best_vanilla_cvae.pth (deflated 8%)
Archives written to:
/content/outputs_vanilla_cvae.zip
/content/checkpoints_vanilla_cvae.zip
